# ECON62020 - Group Replication Project

## Group
Guy Pigott - 8340277
Trinity Rose
Siqi Ge

## Replication Paper

Leora Friedberg, Elliott Isaac, "Same-Sex Marriage Recognition and Taxes: New Evidence about the Impact of Household Taxation", The Review of Economics and Statistics (2024) 106 (1): 85–101.

## Summary

This project replicates key results from Friedberg and Isaac (2024), which examines how marriage-related tax incentives affect the likelihood of same-sex couples marrying. The key variable is the “marriage subsidy” (or penalty), defined as the difference between taxes paid when filing jointly versus separately.

Identification exploits variation in the timing of same-sex marriage recognition across U.S. states, alongside federal recognition following United States v. Windsor (2013). To isolate exogenous variation, the paper constructs a predicted marriage subsidy using LASSO-predicted earnings and uses this as an instrument for the observed subsidy.

This replication reconstructs Table A2 and reproduces the main results in Table 3. We find that higher marriage subsidies increase the probability of being married. While there are differences in levels due to tax simulation approximations, the direction and relative magnitude of the effects are consistent with the original findings.

## Data Description

The analysis uses American Community Survey (ACS) data from 2003–2017, with the unit of observation being a same-sex couple classified as married or cohabiting. Couples are identified using partner links (sploc), sex, and detailed relationship codes, with adjustments for Census recoding.

The dataset includes demographic and income information for both partners. Income components include wages, business income, investment income, retirement income, Social Security income, and transfers, which are used to construct inputs for the NBER TAXSIM model.

The main outcome is an indicator for whether a couple is married. The key explanatory variable is the marriage subsidy, defined as the difference between joint and individual tax liabilities. Two measures are constructed: an observed subsidy based on reported income and a predicted subsidy based on LASSO-predicted earnings.

Control variables include age, education, differences between partners, number of children, race composition, and gender. The specification also includes state and year fixed effects, along with policy variables capturing marriage recognition and Medicaid expansion.

Due to data limitations, some tax inputs are approximated, which may lead to differences in levels relative to the original paper.

## Data Preparation

The data are constructed in two stages. First, a subset of variables is used to identify households containing same-sex couples using partner links and relationship codes. Second, a richer set of variables is extracted for these households only, and couples are formed by matching household heads to partners.

The sample is restricted to individuals aged 18–60 in U.S. states, excluding observations with allocated relationship values. Key variables are then constructed, including total earnings, within-couple earnings splits, demographic characteristics, and policy indicators.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import os

os.chdir("/Users/guypigott/python-venv-demo/EDS")
os.getcwd()

PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / "Data"
OUTPUT_DIR = PROJECT_ROOT / "Data" / "Output"

In [2]:
#file_path = "/Users/guypigott/python-venv-demo/Replication Project/Data/Raw/acs_2012-2017.dat"

# Step 1: Read only 5 columns to find (year, serial) of same-sex couple households.
# These columns are sufficient to determine whether a person is in a same-sex couple.
#colspecs_scan = [
 #   (0,   4),    # year
  ##  (6,   14),   # serial
  #  (77,  81),   # pernum
   # (106, 108),  # sploc
   # (126, 130),  # related  (4-digit detailed version)
   # (130, 131),  # sex
   # (269, 270),  # qrelate
#]
#col_names_scan = ["year", "serial", "pernum", "sploc", "related", "sex", "qrelate"]

#print("Step 1: scanning for same-sex couples...")
#df_scan = pd.read_fwf(file_path, colspecs=colspecs_scan, names=col_names_scan)
#print(f"  Total rows: {len(df_scan):,}")

# ── Identify same-sex couples using vectorised operations ──────────────────────────
# Core logic: find partner via sploc, then compare sex values.

# Sort for consistent merging
#df_scan = df_scan.sort_values(["year","serial","pernum"]).reset_index(drop=True)

# Build a lookup table: (year, serial, pernum) → sex / related / qrelate
# Use merge instead of apply (much faster)
#partner_info = df_scan[["year","serial","pernum","sex","related","qrelate"]].copy()
#partner_info.columns = ["year","serial","sploc","partner_sex","partner_related","partner_qrelate"]

##df_scan = df_scan.merge(partner_info, on=["year","serial","sploc"], how="left")

# Same-sex flag
#same_sex    = df_scan["sex"] == df_scan["partner_sex"]
#sploc_valid = df_scan["sploc"] > 0

# Same-sex married: head (101) + spouse (201), or vice versa
#ss_mar = same_sex & sploc_valid & (
   # ((df_scan["related"] == 101)  & (df_scan["partner_related"] == 201)) |
  #  ((df_scan["related"] == 201)  & (df_scan["partner_related"] == 101))
#)

## Same-sex cohabiting: head (101) + unmarried partner (1114), or vice versa
# ss_coh = same_sex & sploc_valid & (
   # ((df_scan["related"] == 101)  & (df_scan["partner_related"] == 1114)) |
   # ((df_scan["related"] == 1114) & (df_scan["partner_related"] == 101))
#)

# mflag correction: qrelate==9 indicates the Census Bureau re-coded a same-sex
# spouse as "unmarried partner"; these should be treated as married.
#mflag = (df_scan["related"] == 1114) & (df_scan["qrelate"] == 9) & ss_coh

# Propagate mflag to the partner as well
#mflag_info = df_scan[["year","serial","pernum"]].copy()
#mflag_info["mflag"] = mflag.values
#mflag_info.columns = ["year","serial","sploc","partner_mflag"]
#df_scan = df_scan.merge(mflag_info, on=["year","serial","sploc"], how="left")
#df_scan["partner_mflag"] = df_scan["partner_mflag"].fillna(False)

#ss_mar = ss_mar | mflag | df_scan["partner_mflag"]
#ss_coh = ss_coh & ~mflag & ~df_scan["partner_mflag"]

#df_scan["ss_all"] = (ss_mar | ss_coh).astype(int)

# Collect the set of (year, serial) keys that belong to SS households
#ss_keys = df_scan.loc[df_scan["ss_all"] == 1, ["year","serial"]].drop_duplicates()
#print(f"  Same-sex couple households found: {len(ss_keys):,}")
#print(f"  Same-sex individuals: {df_scan['ss_all'].sum():,}")

# Core modification: Add the educd (160, 163) field directly to avoid reading the full file twice.
#colspecs_full = [
#    (0, 4), (6, 14), (37, 39), (77, 81), (106, 108), (118, 119), 
#    (124, 126), (126, 130), (130, 131), (131, 134), (134, 135), 
#    (147, 148), (151, 152), (158, 160), (160, 163), (193, 194), 
#    (251, 258), (261, 262), (269, 270)
#]
#col_names_full = [
 #   "year", "serial", "statefip", "pernum", "sploc", "nchild", 
#    "relate", "related", "sex", "age", "marst", 
#    "race", "hispan", "educ", "educd", "uhrswork", 
#    "incearn", "migrate1", "qrelate"
#]

# Create a hash set to speed up lookups
#ss_set = set(zip(ss_keys["year"], ss_keys["serial"]))
#chunks = []
#CHUNK = 500_000

#print("Step 2: reading full columns for SS households only...")
#reader = pd.read_fwf(file_path, colspecs=colspecs_full, names=col_names_full, chunksize=CHUNK)

#for chunk in reader:
#    mask = [(y, s) in ss_set for y, s in zip(chunk["year"], chunk["serial"])]
#   filtered = chunk[mask]
#   if len(filtered) > 0:
#        chunks.append(filtered)

#df_full = pd.concat(chunks, ignore_index=True)


# Calculate the number of adults in the household (used for subsequent precise filtering of cohabiting couples).
#df_full["adult"] = ((df_full["age"] >= 18) | (df_full["sploc"] != 0)).astype(int)
#num_adults = df_full.groupby(["year","serial"])["adult"].sum().reset_index(name="num_adults")
#df_full = df_full.merge(num_adults, on=["year","serial"], how="left")

#print(f"Done. Full data for SS households: {len(df_full):,} rows")

# Sort data to ensure merge logic is accurate
#df_clean = df_full.sort_values(["year","serial","pernum"]).reset_index(drop=True)

# Extract potential partner info and merge it back to the main table
#p_info = df_clean[["year","serial","pernum","sex","related","qrelate","marst"]].copy()
#p_info.columns = ["year","serial","sploc","p_sex","p_related","p_qrelate","p_marst"]
#df_clean = df_clean.merge(p_info, on=["year","serial","sploc"], how="left")

# Basic conditions: Same sex and valid spouse location (sploc)
#same_sex = df_clean["sex"] == df_clean["p_sex"]
#sploc_valid = df_clean["sploc"] > 0

# Identify same-sex married (ss_mar) and same-sex cohabiting (ss_coh)
#ss_mar = same_sex & sploc_valid & (((df_clean["related"] == 101) & (df_clean["p_related"] == 201)) | ((df_clean["related"] == 201) & (df_clean["p_related"] == 101)))
#ss_coh = same_sex & sploc_valid & (((df_clean["related"] == 101) & (df_clean["p_related"] == 1114)) | ((df_clean["related"] == 1114) & (df_clean["p_related"] == 101)))

# Handle mflag (married couples identified by qrelate==9)
#mflag = (df_clean["related"] == 1114) & (df_clean["qrelate"] == 9) & ss_coh
#mflag_info = df_clean[["year","serial","pernum"]].copy()
#mflag_info["mflag"] = mflag.values
#mflag_info.columns = ["year","serial","sploc","p_mflag"]

#df_clean = df_clean.merge(mflag_info, on=["year","serial","sploc"], how="left")
#df_clean["p_mflag"] = df_clean["p_mflag"].fillna(False)


# Finalize marriage and cohabitation classification
#ss_mar_final = ss_mar | mflag | df_clean["p_mflag"]
#ss_coh_final = ss_coh & ~mflag & ~df_clean["p_mflag"]

#df_clean["sscouple_mar"] = ss_mar_final.astype(int)
#df_clean["sscouple_coh"] = ss_coh_final.astype(int)
#df_clean["ss_any"] = (ss_mar_final | ss_coh_final).astype(int)

# Base filters: Age 18-60, exclude territories, exclude cases with allocated relationships
#mask_base = (
#    (df_clean["ss_any"] == 1) & 
#    (df_clean["age"] >= 18) & (df_clean["age"] <= 60) & 
#    (df_clean["statefip"] <= 56) & 
#    (df_clean["qrelate"] != 4) & (df_clean.get("p_qrelate", 0) != 4)
#)
#df_valid = df_clean[mask_base].copy()

# ================= Assemble Samples =================
# Married logic: Head (101) and their spouse
# mc = df_valid[df_valid["sscouple_mar"] == 1].copy()
#head_m = mc[mc["related"] == 101]
#spouse_m = mc[mc["related"].isin([201, 1114])]
#spouse_m = spouse_m.sort_values("pernum").groupby(["year","serial"]).first().reset_index()
#df_mar = head_m.merge(spouse_m, on=["year","serial"], suffixes=("_h","_s"))
#df_mar["married"] = 1

# Cohabiting logic: To reach 21,136, we typically do not restrict num_adults
#cc = df_valid[df_valid["sscouple_coh"] == 1].copy()
#head_c = cc[cc["related"] == 101]
#partner_c = cc[cc["related"] == 1114]
#df_coh = head_c.merge(partner_c, on=["year","serial"], suffixes=("_h","_s"))
#df_coh["married"] = 0

# Ensure bidirectional sploc matching (excludes complex cohabitation structures)
#df_coh = df_coh[(df_coh["sploc_h"] == df_coh["pernum_s"]) & (df_coh["sploc_s"] == df_coh["pernum_h"])]

# Concatenate and drop duplicates
#df_final = pd.concat([df_mar, df_coh], axis=0, ignore_index=True)
#df_final = df_final.drop_duplicates(subset=["year", "serial"])

#print(f"=== Final Sample (Target Alignment) ===")
#print(f"Married:    {len(df_final[df_final['married']==1]):,} (Paper: 16,098)")
#print(f"Cohabiting: {len(df_final[df_final['married']==0]):,} (Paper: 21,136)")

#Save Cleaned Data
#output_dir = Path("/Users/guypigott/python-venv-demo/EDS/Data/Output")
#output_dir.mkdir(parents=True, exist_ok=True)

#cleaned_file = output_dir / "cleaned_data.pkl"
##df_final.to_pickle(cleaned_file)

#print(f"Saved cleaned data to: {cleaned_file}")
#print(f"Shape: {df_final.shape}")

#csv_file = output_dir / "cleaned_data.csv"
#df_final.to_csv(csv_file, index=False)
#print(f"Saved CSV copy to: {csv_file}")

In [3]:
cleaned_data_path = OUTPUT_DIR / "cleaned_data.pkl"
lasso_input_path = OUTPUT_DIR / "acs_ssc_lasso_input_final.pkl"

df_final = pd.read_pickle(cleaned_data_path)

# Vectorized conversion: educd (detailed education code) to years of education
def educd_to_yrs_vec(s):
    s = s.fillna(-1).astype(int)
    result = np.full(len(s), np.nan)
    result[s <= 12] = 0
    result[s.isin([14, 15, 16, 17])] = s[s.isin([14, 15, 16, 17])] - 13
    result[s.isin([22, 23])] = s[s.isin([22, 23])] - 17
    result[s.isin([25, 26])] = s[s.isin([25, 26])] - 18
    result[s == 30] = 9
    result[s == 40] = 10
    result[s == 50] = 11
    result[(s >= 61) & (s <= 65)] = 12
    result[s == 71] = 13
    result[s == 81] = 14
    result[s == 101] = 16
    result[s >= 114] = 18
    return result

df_f = df_final.copy()

In [4]:
# Income and earnings split calculation
df_f["earn_h"] = df_f["incearn_h"].fillna(0).clip(lower=0)
df_f["earn_s"] = df_f["incearn_s"].fillna(0).clip(lower=0)
df_f["total_earn"] = df_f["earn_h"] + df_f["earn_s"]
df_f["earn_split"] = np.where(df_f["total_earn"] > 0, df_f[["earn_h","earn_s"]].max(axis=1) / df_f["total_earn"], np.nan)

# Construction of demographics and employment status variables
df_f["male"] = (df_f["sex_h"] == 1).astype(int)
df_f["female"] = (df_f["sex_h"] == 2).astype(int)
df_f["age_old"] = np.maximum(df_f["age_h"], df_f["age_s"])
df_f["age_yng"] = np.minimum(df_f["age_h"], df_f["age_s"])
df_f["age_diff"] = df_f["age_old"] - df_f["age_yng"]

df_f["both_work"] = ((df_f["earn_h"]>0) & (df_f["earn_s"]>0)).astype(int)
df_f["one_work"] = ((df_f["earn_h"]>0) ^ (df_f["earn_s"]>0)).astype(int)
df_f["none_work"] = ((df_f["earn_h"]==0) & (df_f["earn_s"]==0)).astype(int)

# Racial matching and dependent children identification
df_f["same_race"] = ((df_f["race_h"] == df_f["race_s"]) & ((df_f["hispan_h"] > 0) == (df_f["hispan_s"] > 0))).astype(int)
df_f["any_children"] = (df_f["nchild_h"] > 0).astype(int)

# Application of education years
df_f["edu_h_yrs"] = educd_to_yrs_vec(df_f["educd_h"])
df_f["edu_s_yrs"] = educd_to_yrs_vec(df_f["educd_s"])
df_f["edu_max"] = np.maximum(df_f["edu_h_yrs"], df_f["edu_s_yrs"])
df_f["edu_min"] = np.minimum(df_f["edu_h_yrs"], df_f["edu_s_yrs"])
df_f["edu_diff"] = df_f["edu_max"] - df_f["edu_min"]


# Output results comparison table
m = df_f[df_f["married"]==1]
c = df_f[df_f["married"]==0]

In [5]:
# ── helper maps ──────────────────────────────────────────────────
def map_education_group(yrs):
    out = np.full(len(yrs), np.nan)
    out[yrs <  12] = 1
    out[yrs == 12] = 2
    out[(yrs > 12) & (yrs < 16)] = 3
    out[yrs >= 16] = 4
    return out

def map_age_group(age):
    out = np.full(len(age), np.nan)
    for upper in range(20, 105, 5):
        mask = (age <= upper) & (age > upper - 5)
        out[mask] = upper
    return out

def map_race(race, hispan):
    out = np.full(len(race), np.nan)
    out[(race == 2) & (hispan == 0)] = 1
    out[((race >= 4) & (race <= 6)) & (hispan == 0)] = 2
    out[((race == 3) | (race >= 7)) & (hispan == 0)] = 3
    out[(race == 1) & (hispan == 0)] = 4
    out[hispan > 0] = 5
    return out

def educd_to_yrs(s):
    s = pd.array(s, dtype="Int64")
    result = np.full(len(s), np.nan)
    result[s <= 12] = 0
    result[s == 14] = 1
    result[s == 15] = 2
    result[s == 16] = 3
    result[s == 17] = 4
    result[s == 22] = 5
    result[s == 23] = 6
    result[s == 25] = 7
    result[s == 26] = 8
    result[s == 30] = 9
    result[s == 40] = 10
    result[s == 50] = 11
    result[(s >= 61) & (s <= 65)] = 12
    result[s == 71] = 13
    result[s == 81] = 14
    result[s == 101] = 16
    result[s >= 114] = 18
    return result

# ── build output dataframe ───────────────────────────────────────
d = df_final.copy()
d["uhrswork_h"] = pd.to_numeric(d["uhrswork_h"], errors="coerce").fillna(0)
d["uhrswork_s"] = pd.to_numeric(d["uhrswork_s"], errors="coerce").fillna(0)
d["incearn_h"] = d["incearn_h"].clip(lower=0)
d["incearn_s"] = d["incearn_s"].clip(lower=0)

out = pd.DataFrame()
out["year"]         = d["year"]
out["serial"]       = d["serial"]
out["statefip"]     = d["statefip_h"]
out["sscouple_mar"] = d["married"].astype(bool)
out["sscouple_coh"] = (d["married"] == 0).astype(bool)
out["sscouple_all"] = True
out["r_male"]       = (d["sex_h"] == 1).astype(int)
out["age"]          = d["age_h"]
out["r_incearn"]    = d["incearn_h"]
r_yrs               = educd_to_yrs(d["educd_h"].fillna(-1).astype(int).values)
out["r_yrsedu"]     = r_yrs
out["r_edugroup"]   = map_education_group(r_yrs)
out["r_agegroup"]   = map_age_group(d["age_h"].values)
out["r_race"]       = map_race(d["race_h"].values, d["hispan_h"].values)
out["r_occbroad"]   = 0
out["r_degbroad"]   = 0
out["sp_male"]      = (d["sex_s"] == 1).astype(int)
out["sp_age"]       = d["age_s"]
out["sp_incearn"]   = d["incearn_s"]
sp_yrs              = educd_to_yrs(d["educd_s"].fillna(-1).astype(int).values)
out["sp_yrsedu"]    = sp_yrs
out["sp_edugroup"]  = map_education_group(sp_yrs)
out["sp_agegroup"]  = map_age_group(d["age_s"].values)
out["sp_race"]      = map_race(d["race_s"].values, d["hispan_s"].values)
out["sp_occbroad"]  = 0
out["sp_degbroad"]  = 0
out["c_children"]   = d["nchild_h"]
out["c_anychildren"]= (d["nchild_h"] > 0).astype(int)
out["c_racecomp"]   = (out["r_race"] == out["sp_race"]).astype(int)
out["c_agemax"]     = np.maximum(d["age_h"], d["age_s"])
out["c_agemin"]     = np.minimum(d["age_h"], d["age_s"])
out["c_agediff"]    = out["c_agemax"] - out["c_agemin"]
out["c_edumax"]     = np.maximum(r_yrs, sp_yrs)
out["c_edumin"]     = np.minimum(r_yrs, sp_yrs)
out["c_edudiff"]    = out["c_edumax"] - out["c_edumin"]
out["c_incearn"]    = d["incearn_h"] + d["incearn_s"]
total               = out["c_incearn"].replace(0, np.nan)
out["c_incearn_split"] = (np.maximum(d["incearn_h"], d["incearn_s"]) / total).fillna(0.5)
for i in range(1, 6):
    out[f"c_incearn{i}"]       = (out["c_incearn"] / 10_000) ** i
    out[f"c_incearn_split{i}"] =  out["c_incearn_split"] ** i
out["r_posincearn"]  = (d["incearn_h"] > 0).astype(int)
out["sp_posincearn"] = (d["incearn_s"] > 0).astype(int)
out["c_dualearner"]  = ((d["incearn_h"] > 0) & (d["incearn_s"] > 0)).astype(int)
out["c_singleearner"]= ((d["incearn_h"] > 0) ^ (d["incearn_s"] > 0)).astype(int)
out["c_noearner"]    = ((d["incearn_h"] == 0) & (d["incearn_s"] == 0)).astype(int)
out["migrate1"]      = d["migrate1_h"]

# ── policy columns ───────────────────────────────────────────────
ssm_start = {
    2: 2014, 4: 2014, 6: 2013, 8: 2014, 9: 2008, 10: 2013, 11: 2010,
    12: 2015, 15: 2013, 16: 2014, 17: 2014, 18: 2014, 19: 2009,
    23: 2012, 24: 2013, 25: 2004, 27: 2013, 30: 2014, 32: 2014,
    33: 2010, 34: 2013, 35: 2013, 36: 2011, 37: 2014, 40: 2014,
    41: 2014, 42: 2014, 44: 2013, 45: 2014, 49: 2014, 50: 2009,
    51: 2014, 53: 2012, 54: 2014, 55: 2014, 56: 2014
}
out["staterecog_policy"] = (
    out["year"] >= out["statefip"].map(ssm_start).fillna(2015)
).astype(int)
out["preW"]      = (out["year"] <= 2012).astype(int)
out["postWpreO"] = (out["year"].between(2013, 2014)).astype(int)
out["postO"]     = (out["year"] >= 2015).astype(int)

MEDICAID_EXP_STATES = {
    4, 6, 8, 9, 10, 11, 15, 17, 18, 19, 21, 23, 24, 25,
    26, 27, 32, 33, 34, 35, 36, 38, 41, 44, 50, 53, 55
}
out["medicaid_exp"] = (
    (out["year"] >= 2014) & (out["statefip"].isin(MEDICAID_EXP_STATES))
).astype(int)

# ── sanity check & save ──────────────────────────────────────────
print(f"Output shape: {out.shape}")
print(f"Policy columns: {['staterecog_policy','preW','postWpreO','postO','medicaid_exp']}")
out.to_pickle(lasso_input_path)

Output shape: (37907, 56)
Policy columns: ['staterecog_policy', 'preW', 'postWpreO', 'postO', 'medicaid_exp']


## Results - Table A2 

Table A2 reports summary statistics for the sample of same-sex couples used in the analysis, split between married and cohabiting couples. The table is designed to demonstrate that the constructed dataset closely matches the sample and descriptive statistics reported in the original paper.

In [6]:
print("=== Table A2 Full Comparison ===\n")
print(f"{'Variable':<35} {'Our Married':>18} {'Our Cohab':>18} {'Paper Married':>14} {'Paper Cohab':>12}")
print("─"*100)

def row(label, s1, s2, pm, pc):
    v1 = f"{np.nanmean(s1):.3f} ({np.nanstd(s1):.3f})"
    v2 = f"{np.nanmean(s2):.3f} ({np.nanstd(s2):.3f})"
    print(f"{label:<35} {v1:>18} {v2:>18} {pm:>14} {pc:>12}")

row("Male", m["male"], c["male"], "0.469 (0.499)", "0.506 (0.500)")
row("Female", m["female"], c["female"], "0.531 (0.499)", "0.494 (0.500)")
row("Same race", m["same_race"], c["same_race"], "0.793 (0.405)", "0.757 (0.429)")
row("Age older", m["age_old"], c["age_old"], "46.205 (9.633)","43.353 (10.902)")
row("Age younger", m["age_yng"], c["age_yng"], "41.252 (9.831)","37.858 (10.442)")
row("Age diff", m["age_diff"], c["age_diff"], "4.953 (5.234)", "5.495 (5.543)")
row("Edu max (yrs)", m["edu_max"], c["edu_max"], "15.594 (2.468)","15.363 (2.264)")
row("Edu min (yrs)", m["edu_min"], c["edu_min"], "13.718 (3.044)","13.513 (2.604)")
row("Edu diff", m["edu_diff"], c["edu_diff"], "1.875 (2.286)", "1.850 (2.108)")
row("Any children", m["any_children"], c["any_children"], "0.314 (0.464)","0.162 (0.369)")
row("Both work", m["both_work"], c["both_work"], "0.776 (0.417)", "0.811 (0.392)")
row("Only 1 works", m["one_work"], c["one_work"], "0.195 (0.396)", "0.160 (0.367)")
row("Neither works", m["none_work"], c["none_work"], "0.029 (0.167)", "0.029 (0.168)")
row("Total earnings", m["total_earn"], c["total_earn"], "125287 (119780)","105188 (105192)")
row("Earnings split", m.loc[m["total_earn"]>0,"earn_split"], c.loc[c["total_earn"]>0,"earn_split"], "0.745 (0.200)", "0.723 (0.174)")
print("─"*100)
print(f"{'Observations':<35} {len(m):>18,} {len(c):>18,} {'16,098':>14} {'21,136':>12}")

=== Table A2 Full Comparison ===

Variable                                   Our Married          Our Cohab  Paper Married  Paper Cohab
────────────────────────────────────────────────────────────────────────────────────────────────────
Male                                     0.468 (0.499)      0.497 (0.500)  0.469 (0.499) 0.506 (0.500)
Female                                   0.532 (0.499)      0.503 (0.500)  0.531 (0.499) 0.494 (0.500)
Same race                                0.783 (0.412)      0.744 (0.436)  0.793 (0.405) 0.757 (0.429)
Age older                               46.209 (9.634)    43.251 (10.933) 46.205 (9.633) 43.353 (10.902)
Age younger                             41.241 (9.842)    37.698 (10.462) 41.252 (9.831) 37.858 (10.442)
Age diff                                 4.968 (5.263)      5.554 (5.690)  4.953 (5.234) 5.495 (5.543)
Edu max (yrs)                           15.588 (2.472)     15.345 (2.264) 15.594 (2.468) 15.363 (2.264)
Edu min (yrs)                        

The resulting sample closely matches the original paper. We obtain 16,146 married and 21,761 cohabiting couples, compared to 16,098 and 21,136 in the paper. Demographic characteristics, labour supply measures, and income variables align almost exactly. For example, average earnings and earnings shares are nearly identical across both datasets.

Minor differences arise in variables such as the share of couples with children, likely due to small differences in variable construction. Overall, the close match provides strong validation of the data preparation process and supports the reliability of subsequent analysis.

## Results - Table 3

To address endogeneity in observed earnings, we implement the paper’s two-step LASSO procedure. First, LASSO is used to predict labour force participation. Second, conditional on participation, LASSO predicts individual earnings using demographic characteristics. The model is trained on 2012 data and applied to other years.

Predicted earnings are aggregated to the couple level and used as inputs into the TAXSIM model (version 35) to simulate tax liabilities under joint and separate filing. The difference defines the predicted marriage subsidy, which depends only on exogenous characteristics.

We then estimate the causal effect of tax incentives using an instrumental variables approach, where the predicted subsidy instruments for the observed subsidy. This isolates variation in marriage incentives driven by the tax system rather than behavioural responses, allowing for a causal interpretation of the effect of financial incentives on marriage decisions.

In [7]:
from sklearn.linear_model import LassoCV
from sklearn.preprocessing import StandardScaler

Predicted_output  = OUTPUT_DIR / "acs_ssc_predicted_second.pkl"

df_lasso = pd.read_pickle(lasso_input_path)
print(f"Loaded: {df_lasso.shape}  |  Years: {sorted(df_lasso['year'].unique())}")


Loaded: (37907, 56)  |  Years: [np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017)]


In [8]:
# Confirm all required feature columns are present
needed = [
    "r_agegroup", "r_edugroup", "r_race", "r_occbroad", "r_degbroad", "r_male",
    "sp_agegroup", "sp_edugroup", "sp_race", "sp_occbroad", "sp_degbroad", "sp_male",
    "c_children", "statefip", "r_incearn", "sp_incearn"
]
missing = [c for c in needed if c not in df_lasso.columns]
if missing:
    raise ValueError(f"Still missing columns: {missing}")
print("All columns present.")

All columns present.


In [9]:
# ── Feature builder ──────────────────────────────────────────────
def build_features(person_df: pd.DataFrame) -> pd.DataFrame:
    """
    Expects columns already renamed to the canonical names:
      r_agegroup, r_edugroup, r_race, r_occbroad, r_degbroad,
      r_male, statefip, c_children
    Returns dummy-expanded + pairwise-interaction matrix.
    """
    cat_cols = ["r_agegroup", "r_edugroup", "r_race",
                "r_occbroad", "r_degbroad", "statefip"]

    base = pd.get_dummies(
        person_df[cat_cols + ["r_male", "c_children"]],
        columns=cat_cols,
        drop_first=True,
        dtype=float
    )

    # Pairwise interactions — keep only non-trivial (sum >= 10)
    base_cols = list(base.columns)
    interact_parts = []
    for i, c1 in enumerate(base_cols):
        for c2 in base_cols[i + 1:]:
            s = base[c1] * base[c2]
            if s.sum() >= 10:
                interact_parts.append(s.rename(f"{c1}_x_{c2}"))

    if interact_parts:
        X = pd.concat([base, pd.concat(interact_parts, axis=1)], axis=1)
    else:
        X = base

    return X.astype(float)


print("Feature builder defined.")

Feature builder defined.


In [10]:
def predict_income_for_role(df, train_mask, prefix="r", cv=10, max_iter=10_000):
    earn_col = f"{prefix}_incearn"

    X_full = build_features(df)
    scaler = StandardScaler()
    X_sc = scaler.fit_transform(X_full)
    X_train = X_sc[train_mask.values]

    # Step 1: P(earn > 0)
    y1 = (df[earn_col] > 0).astype(float).values
    y1_train = y1[train_mask.values]

    m1 = LassoCV(cv=cv, max_iter=max_iter, n_jobs=-1, random_state=42)
    m1.fit(X_train, y1_train)

    hat_pos_prob = m1.predict(X_sc)

    # Set threshold 
    target_share = y1.mean()                          
    threshold = np.percentile(hat_pos_prob, (1 - target_share) * 100)
    hat_pos = (hat_pos_prob >= threshold).astype(float)

    print(f"  [{prefix}] target share: {target_share:.3f}  "
      f"threshold: {threshold:.3f}  "
      f"achieved share: {hat_pos.mean():.3f}")

    # Step 2: earn in levels | earn > 0
    pos_train = train_mask.values & (df[earn_col].values > 0)
    y2_train = df.loc[pos_train, earn_col].values
    X2_train = X_sc[pos_train]

    m2 = LassoCV(cv=cv, max_iter=max_iter, n_jobs=-1, random_state=42)
    m2.fit(X2_train, y2_train)

    hat_earn_lvl = np.maximum(m2.predict(X_sc), 0)
    hat_incearn = np.where(hat_pos == 1, hat_earn_lvl, 0.0)

    df = df.copy()
    df[f"hat_pos_{prefix}"]     = hat_pos
    df[f"hat_incearn_{prefix}"] = hat_incearn
    return df


In [11]:
# Find the threshold 
y1_r = (df_lasso["r_incearn"] > 0).astype(float).values
train_mask = df_lasso["year"] == 2012

m1_temp = LassoCV(cv=10, max_iter=10_000, n_jobs=-1, random_state=42)
X_full = build_features(df_lasso)
scaler = StandardScaler()
X_sc = scaler.fit_transform(X_full)
m1_temp.fit(X_sc[train_mask.values], y1_r[train_mask.values])
hat_pos_prob_r = m1_temp.predict(X_sc)

paper_target_r  = 0.963   # from Table 2
paper_target_sp = 0.969

# Find threshold that achieves paper target
for t in np.arange(0.77, 0.95, 0.01):
    achieved = (hat_pos_prob_r >= t).mean()
    print(f"threshold={t:.2f}  achieved share={achieved:.3f}  target={paper_target_r:.3f}  diff={achieved-paper_target_r:+.3f}")

threshold=0.77  achieved share=0.988  target=0.963  diff=+0.025
threshold=0.78  achieved share=0.957  target=0.963  diff=-0.006
threshold=0.79  achieved share=0.954  target=0.963  diff=-0.009
threshold=0.80  achieved share=0.948  target=0.963  diff=-0.015
threshold=0.81  achieved share=0.943  target=0.963  diff=-0.020
threshold=0.82  achieved share=0.916  target=0.963  diff=-0.047
threshold=0.83  achieved share=0.903  target=0.963  diff=-0.060
threshold=0.84  achieved share=0.804  target=0.963  diff=-0.159
threshold=0.85  achieved share=0.789  target=0.963  diff=-0.174
threshold=0.86  achieved share=0.679  target=0.963  diff=-0.284
threshold=0.87  achieved share=0.663  target=0.963  diff=-0.300
threshold=0.88  achieved share=0.605  target=0.963  diff=-0.358
threshold=0.89  achieved share=0.488  target=0.963  diff=-0.475
threshold=0.90  achieved share=0.456  target=0.963  diff=-0.507
threshold=0.91  achieved share=0.452  target=0.963  diff=-0.511
threshold=0.92  achieved share=0.448  ta

In [12]:
y1_sp = (df_lasso["sp_incearn"] > 0).astype(float).values

m1_temp_sp = LassoCV(cv=10, max_iter=10_000, n_jobs=-1, random_state=42)
X_sc_sp = scaler.fit_transform(X_full)  # reuse X_full and scaler from above
m1_temp_sp.fit(X_sc_sp[train_mask.values], y1_sp[train_mask.values])
hat_pos_prob_sp = m1_temp_sp.predict(X_sc_sp)

paper_target_sp = 0.969

for t in np.arange(0.77, 0.95, 0.01):
    achieved = (hat_pos_prob_sp >= t).mean()
    print(f"threshold={t:.2f}  achieved share={achieved:.3f}  target={paper_target_sp:.3f}  diff={achieved-paper_target_sp:+.3f}")

threshold=0.77  achieved share=0.923  target=0.969  diff=-0.046
threshold=0.78  achieved share=0.910  target=0.969  diff=-0.059
threshold=0.79  achieved share=0.891  target=0.969  diff=-0.078
threshold=0.80  achieved share=0.864  target=0.969  diff=-0.105
threshold=0.81  achieved share=0.843  target=0.969  diff=-0.126
threshold=0.82  achieved share=0.803  target=0.969  diff=-0.166
threshold=0.83  achieved share=0.763  target=0.969  diff=-0.206
threshold=0.84  achieved share=0.666  target=0.969  diff=-0.303
threshold=0.85  achieved share=0.620  target=0.969  diff=-0.349
threshold=0.86  achieved share=0.521  target=0.969  diff=-0.448
threshold=0.87  achieved share=0.447  target=0.969  diff=-0.522
threshold=0.88  achieved share=0.407  target=0.969  diff=-0.562
threshold=0.89  achieved share=0.305  target=0.969  diff=-0.664
threshold=0.90  achieved share=0.273  target=0.969  diff=-0.696
threshold=0.91  achieved share=0.136  target=0.969  diff=-0.833
threshold=0.92  achieved share=0.054  ta

In [13]:
def predict_income_for_role(df, train_mask, prefix="r", cv=10, max_iter=10_000):
    earn_col = f"{prefix}_incearn"

    X_full = build_features(df)
    scaler = StandardScaler()
    X_sc = scaler.fit_transform(X_full)
    X_train = X_sc[train_mask.values]

    y1 = (df[earn_col] > 0).astype(float).values
    y1_train = y1[train_mask.values]

    m1 = LassoCV(cv=cv, max_iter=max_iter, n_jobs=-1, random_state=42)
    m1.fit(X_train, y1_train)

    hat_pos_prob = m1.predict(X_sc)

    # Thresholds
    threshold = 0.78 if prefix == "r" else 0.77
    hat_pos = (hat_pos_prob >= threshold).astype(float)

    print(f"  [{prefix}] threshold={threshold}  "
          f"achieved={hat_pos.mean():.3f}  "
          f"paper target={'0.963' if prefix=='r' else '0.969'}")

    pos_train = train_mask.values & (df[earn_col].values > 0)
    y2_train = df.loc[pos_train, earn_col].values
    X2_train = X_sc[pos_train]

    m2 = LassoCV(cv=cv, max_iter=max_iter, n_jobs=-1, random_state=42)
    m2.fit(X2_train, y2_train)

    hat_earn_lvl = np.maximum(m2.predict(X_sc), 0)
    hat_incearn = np.where(hat_pos == 1, hat_earn_lvl, 0.0)

    df = df.copy()
    df[f"hat_pos_{prefix}"]     = hat_pos
    df[f"hat_incearn_{prefix}"] = hat_incearn
    return df

In [14]:
# ── Run — Respondent ────────────────────────────────────────────
train_mask = df_lasso["year"] == 2012
print(f"Training sample (2012): N={train_mask.sum():,}")

df = predict_income_for_role(df_lasso, train_mask, prefix="r")

Training sample (2012): N=5,070
  [r] threshold=0.78  achieved=0.957  paper target=0.963


In [15]:
# ── Run — Spouse / partner ──────────────────────────────────────
df = predict_income_for_role(df, train_mask, prefix="sp")

  [sp] threshold=0.77  achieved=0.923  paper target=0.969


In [16]:
# ── Couple-level aggregates ──────────────────────────────────────
df["hat_c_incearn"] = df["hat_incearn_r"] + df["hat_incearn_sp"]

# Primary earner share = max(r, sp) / total  (matches paper's Table 2)
total = df["hat_c_incearn"].replace(0, np.nan)
df["hat_c_split"] = (
    np.maximum(df["hat_incearn_r"], df["hat_incearn_sp"]) / total
).fillna(0.5)

# Store the 5-polynomial terms
for i in range(1, 6):
    df[f"hat_c_incearn{i}"] = (df["hat_c_incearn"] / 10_000) ** i
    df[f"hat_c_split{i}"]   = df["hat_c_split"] ** i

print("Couple-level columns added.")

Couple-level columns added.


In [17]:
# ── Diagnostics vs Table 2 ───────────────────────────────────────
print("="*60)
print("DIAGNOSTICS vs. Table 2")
print("="*60)

print("\n--- Share positive earnings ---")
print(f"  Respondent  reported={(df['r_incearn']>0).mean():.3f}  "
      f"predicted={df['hat_pos_r'].mean():.3f}  "
      f"(paper predicted: 0.963 mar / 0.969 coh)")
print(f"  Spouse      reported={(df['sp_incearn']>0).mean():.3f}  "
      f"predicted={df['hat_pos_sp'].mean():.3f}")

print("\n--- Couple predicted total earnings ---")
for label, mask in [("Married", df["sscouple_mar"]),
                    ("Cohabiting", df["sscouple_coh"])]:
    sub = df[mask]
    print(f"  {label}: {sub['hat_c_incearn'].mean():>10,.0f}  "
          f"(paper: mar≈110,729  coh≈102,953)")

print("\n--- Couple predicted earnings split ---")
for label, mask in [("Married", df["sscouple_mar"]),
                    ("Cohabiting", df["sscouple_coh"])]:
    sub = df[mask]
    pos = sub["hat_c_incearn"] > 0
    print(f"  {label}: {sub.loc[pos,'hat_c_split'].mean():.3f}  "
          f"(paper: mar≈0.648  coh≈0.641)")

print("\n--- Correlation reported vs predicted (positive earners) ---")
for pfx, label in [("r", "Respondent"), ("sp", "Spouse")]:
    both = (df[f"{pfx}_incearn"] > 0) & (df[f"hat_incearn_{pfx}"] > 0)
    r = np.corrcoef(df.loc[both, f"{pfx}_incearn"],
                    df.loc[both, f"hat_incearn_{pfx}"])[0, 1]
    print(f"  {label}: r={r:.3f}")

DIAGNOSTICS vs. Table 2

--- Share positive earnings ---
  Respondent  reported=0.905  predicted=0.957  (paper predicted: 0.963 mar / 0.969 coh)
  Spouse      reported=0.863  predicted=0.923

--- Couple predicted total earnings ---
  Married:    115,730  (paper: mar≈110,729  coh≈102,953)
  Cohabiting:    108,429  (paper: mar≈110,729  coh≈102,953)

--- Couple predicted earnings split ---
  Married: 0.616  (paper: mar≈0.648  coh≈0.641)
  Cohabiting: 0.602  (paper: mar≈0.648  coh≈0.641)

--- Correlation reported vs predicted (positive earners) ---
  Respondent: r=0.416
  Spouse: r=0.304


In [18]:
df.to_pickle(Predicted_output)
print(f"Saved to {Predicted_output}")

Saved to /Users/guypigott/python-venv-demo/EDS/Data/Output/acs_ssc_predicted_second.pkl


In [19]:
import torch
if torch.backends.mps.is_available() and torch.backends.mps.is_built():
    device = torch.device("mps")
    print("MPS device found")
else:
    device = torch.device("cpu")
    print("MPS device not found")

MPS device found


In [20]:
import torch

# 1. Create two large random matrices
# We use float32 as it is the standard for GPU operations
size = 5000
x = torch.randn(size, size, device=device)
y = torch.randn(size, size, device=device)

# 2. Perform matrix multiplication on the GPU
# If this runs without error, your MPS setup is perfect
result = torch.matmul(x, y)

print(f"Calculation complete!")
print(f"Tensor is located on: {result.device}")

Calculation complete!
Tensor is located on: mps:0


In [21]:
import io
import os
import subprocess

# ── Paths ─────────────────────────────────────────────────────────
taxsim_output = OUTPUT_DIR / "acs_ssc_taxsim_results_second.pkl"
TAXSIM_EXE = os.path.expanduser("~/Desktop/taxsim35/taxsim35.exe")

if not os.path.exists(TAXSIM_EXE):
    raise FileNotFoundError(
        f"Binary not found at {TAXSIM_EXE}.\n"
        "Run: curl -o ~/taxsim35 "
        "https://taxsim.nber.org/stata/taxsim35/taxsim35-osx.exe "
        "&& chmod +x ~/taxsim35"
    )

df_pred = pd.read_pickle(Predicted_output)
print(f"Loaded: {df_pred.shape}")
print(f"Years:  {sorted(df_pred['year'].unique())}")
print(f"Married:    {(df_pred['sscouple_mar']==True).sum():,}")
print(f"Cohabiting: {(df_pred['sscouple_mar']==False).sum():,}")

Loaded: (37907, 72)
Years:  [np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017)]
Married:    16,146
Cohabiting: 21,761


In [22]:
# ── FIPS → SOI state code conversion ─────────────────────────────
# TAXSIM uses IRS SOI codes (sequential 1-51), not Census FIPS.

FIPS_TO_SOI = {
    1:  1,  2:  2,  4:  3,  5:  4,  6:  5,  8:  6,  9:  7,
    10: 8,  11: 9,  12: 10, 13: 11, 15: 12, 16: 13, 17: 14,
    18: 15, 19: 16, 20: 17, 21: 18, 22: 19, 23: 20, 24: 21,
    25: 22, 26: 23, 27: 24, 28: 25, 29: 26, 30: 27, 31: 28,
    32: 29, 33: 30, 34: 31, 35: 32, 36: 33, 37: 34, 38: 35,
    39: 36, 40: 37, 41: 38, 42: 39, 44: 40, 45: 41, 46: 42,
    47: 43, 48: 44, 49: 45, 50: 46, 51: 47, 53: 48, 54: 49,
    55: 50, 56: 51,
}

unmapped = set(df_pred["statefip"].unique()) - set(FIPS_TO_SOI.keys())
print(f"Unmapped FIPS: {unmapped if unmapped else 'None'}")

Unmapped FIPS: None


In [23]:
# ── TAXSIM35 chunked runner ───────────────────────────────────────

def run_taxsim35(df_taxsim: pd.DataFrame, chunksize: int = 1000,
                 label: str = "") -> pd.DataFrame:
    chunks_out = []
    n = len(df_taxsim)
    n_chunks = (n + chunksize - 1) // chunksize
    for i in range(n_chunks):
        chunk = df_taxsim.iloc[i * chunksize : (i + 1) * chunksize].copy()
        chunk["taxsimid"] = np.arange(1, len(chunk) + 1)
        result = subprocess.run(
            [TAXSIM_EXE],
            input=chunk.to_csv(index=False),
            capture_output=True, text=True, timeout=300,
        )
        stdout = result.stdout.strip()
        if not stdout or not stdout.startswith("taxsimid"):
            raise RuntimeError(
                f"{label} chunk {i+1}/{n_chunks} failed.\n"
                f"stderr: {result.stderr[:1000]}\n"
                f"stdout: {stdout[:500]}"
            )
        chunks_out.append(pd.read_csv(io.StringIO(result.stdout)))
        if (i + 1) % 10 == 0 or i == n_chunks - 1:
            print(f"  {label}: {i+1}/{n_chunks} chunks "
                  f"({min((i+1)*chunksize, n):,}/{n:,} rows)")
    return pd.concat(chunks_out, ignore_index=True)


# ── Input builders ────────────────────────────────────────────────
# Per paper p.89: 'no other tax expenditures' means depx=0 in ALL runs.
# Children are controlled for separately in the regression.
# Both single filers use mstat=1 (plain single, no HoH).

def make_single(df_pred, wage_col, age_col):
    """Single filer — wages only, no children, plain mstat=1."""
    t = pd.DataFrame()
    t["taxsimid"] = np.arange(1, len(df) + 1)
    t["year"]   = df_pred["year"].values
    t["state"]  = df_pred["statefip"].map(FIPS_TO_SOI)
    t["mstat"]  = 1
    t["page"]   = df_pred[age_col].values
    t["sage"]   = 0
    t["pwages"] = df_pred[wage_col].clip(lower=0).fillna(0).round().astype(int)
    t["swages"] = 0
    t["depx"]   = 0  
    return t


def make_joint(df_pred, wage_r_col, wage_sp_col, age_r_col, age_sp_col):
    """Joint filer — wages only, no children, mstat=2."""
    t = pd.DataFrame()
    t["taxsimid"] = np.arange(1, len(df) + 1)
    t["year"]   = df_pred["year"].values
    t["state"]  = df_pred["statefip"].map(FIPS_TO_SOI)
    t["mstat"]  = 2
    t["page"]   = df_pred[age_r_col].values
    t["sage"]   = df_pred[age_sp_col].values
    t["pwages"] = df_pred[wage_r_col].clip(lower=0).fillna(0).round().astype(int)
    t["swages"] = df_pred[wage_sp_col].clip(lower=0).fillna(0).round().astype(int)
    t["depx"]   = 0   
    return t


print("Helpers defined.")

Helpers defined.


In [24]:
# ══════════════════════════════════════════════════════════════════
# RUN A — PREDICTED INCOME  (instrument for Table 3 IV)
# Uses hat_incearn_r / hat_incearn_sp from LASSO
# ══════════════════════════════════════════════════════════════════

print("Run A1: respondent as single (predicted)...")
a1_out = run_taxsim35(
    make_single(df_pred, "hat_incearn_r", "age"),
    label="A1")

print("Run A2: spouse as single (predicted)...")
a2_out = run_taxsim35(
    make_single(df_pred, "hat_incearn_sp", "sp_age"),
    label="A2")

print("Run A3: couple jointly (predicted)...")
a3_out = run_taxsim35(
    make_joint(df_pred, "hat_incearn_r", "hat_incearn_sp", "age", "sp_age"),
    label="A3")

assert len(a1_out) == len(a2_out) == len(a3_out) == len(df), "Row mismatch!"
print(f"\nA runs done.")
print(f"  A1 mean fiitax: {a1_out['fiitax'].mean():,.2f}")
print(f"  A2 mean fiitax: {a2_out['fiitax'].mean():,.2f}")
print(f"  A3 mean fiitax: {a3_out['fiitax'].mean():,.2f}")
print(f"  Raw fed subsidy (all): {(a1_out['fiitax']+a2_out['fiitax']-a3_out['fiitax']).mean():,.2f}")

Run A1: respondent as single (predicted)...
  A1: 10/38 chunks (10,000/37,907 rows)
  A1: 20/38 chunks (20,000/37,907 rows)
  A1: 30/38 chunks (30,000/37,907 rows)
  A1: 38/38 chunks (37,907/37,907 rows)
Run A2: spouse as single (predicted)...
  A2: 10/38 chunks (10,000/37,907 rows)
  A2: 20/38 chunks (20,000/37,907 rows)
  A2: 30/38 chunks (30,000/37,907 rows)
  A2: 38/38 chunks (37,907/37,907 rows)
Run A3: couple jointly (predicted)...
  A3: 10/38 chunks (10,000/37,907 rows)
  A3: 20/38 chunks (20,000/37,907 rows)
  A3: 30/38 chunks (30,000/37,907 rows)
  A3: 38/38 chunks (37,907/37,907 rows)

A runs done.
  A1 mean fiitax: 9,913.02
  A2 mean fiitax: 6,129.85
  A3 mean fiitax: 15,817.73
  Raw fed subsidy (all): 225.13


In [25]:
print([c for c in df_pred.columns if 'hat' in c or 'incearn' in c or 'lasso' in c.lower()])

['r_incearn', 'sp_incearn', 'c_incearn', 'c_incearn_split', 'c_incearn1', 'c_incearn_split1', 'c_incearn2', 'c_incearn_split2', 'c_incearn3', 'c_incearn_split3', 'c_incearn4', 'c_incearn_split4', 'c_incearn5', 'c_incearn_split5', 'r_posincearn', 'sp_posincearn', 'hat_pos_r', 'hat_incearn_r', 'hat_pos_sp', 'hat_incearn_sp', 'hat_c_incearn', 'hat_c_split', 'hat_c_incearn1', 'hat_c_split1', 'hat_c_incearn2', 'hat_c_split2', 'hat_c_incearn3', 'hat_c_split3', 'hat_c_incearn4', 'hat_c_split4', 'hat_c_incearn5', 'hat_c_split5']


In [26]:
# ══════════════════════════════════════════════════════════════════
# RUN B — REPORTED INCOME  (observed subsidy for Table 2 + OLS)
# Uses r_incearn / sp_incearn directly from ACS
# ══════════════════════════════════════════════════════════════════

print("Run B1: respondent as single (reported)...")
b1_out = run_taxsim35(
    make_single(df_pred, "r_incearn", "age"),
    label="B1")

print("Run B2: spouse as single (reported)...")
b2_out = run_taxsim35(
    make_single(df_pred, "sp_incearn", "sp_age"),
    label="B2")

print("Run B3: couple jointly (reported)...")
b3_out = run_taxsim35(
    make_joint(df_pred, "r_incearn", "sp_incearn", "age", "sp_age"),
    label="B3")

assert len(b1_out) == len(b2_out) == len(b3_out) == len(df), "Row mismatch!"
print(f"\nB runs done.")
print(f"  Raw fed subsidy (all): {(b1_out['fiitax']+b2_out['fiitax']-b3_out['fiitax']).mean():,.2f}")

Run B1: respondent as single (reported)...
  B1: 10/38 chunks (10,000/37,907 rows)
  B1: 20/38 chunks (20,000/37,907 rows)
  B1: 30/38 chunks (30,000/37,907 rows)
  B1: 38/38 chunks (37,907/37,907 rows)
Run B2: spouse as single (reported)...
  B2: 10/38 chunks (10,000/37,907 rows)
  B2: 20/38 chunks (20,000/37,907 rows)
  B2: 30/38 chunks (30,000/37,907 rows)
  B2: 38/38 chunks (37,907/37,907 rows)
Run B3: couple jointly (reported)...
  B3: 10/38 chunks (10,000/37,907 rows)
  B3: 20/38 chunks (20,000/37,907 rows)
  B3: 30/38 chunks (30,000/37,907 rows)
  B3: 38/38 chunks (37,907/37,907 rows)

B runs done.
  Raw fed subsidy (all): 686.62


In [27]:
# ── Compute marriage subsidies ────────────────────────────────────
# Subsidy = (T_i + T_j) - T_c
# Positive = subsidy (married pay less). Negative = penalty.

df_out = df_pred.copy()
m = df_out["sscouple_mar"] == True
c = df_out["sscouple_mar"] == False

# Predicted income subsidies (instrument)
df_out["fed_sub_pred"]   = (a1_out["fiitax"].values + a2_out["fiitax"].values
                            - a3_out["fiitax"].values)
df_out["state_sub_pred"] = (a1_out["siitax"].values + a2_out["siitax"].values
                            - a3_out["siitax"].values)
df_out["total_sub_pred"] = df_out["fed_sub_pred"] + df_out["state_sub_pred"]

# Reported income subsidies (observed)
df_out["fed_sub_obs"]    = (b1_out["fiitax"].values + b2_out["fiitax"].values
                            - b3_out["fiitax"].values)
df_out["state_sub_obs"]  = (b1_out["siitax"].values + b2_out["siitax"].values
                            - b3_out["siitax"].values)
df_out["total_sub_obs"]  = df_out["fed_sub_obs"] + df_out["state_sub_obs"]

# Joint filing totals (for other analyses)
df_out["fiitax_joint_pred"] = a3_out["fiitax"].values
df_out["siitax_joint_pred"] = a3_out["siitax"].values
df_out["fica_joint_pred"]   = a3_out["fica"].values
df_out["fiitax_joint_obs"]  = b3_out["fiitax"].values
df_out["siitax_joint_obs"]  = b3_out["siitax"].values
df_out["fica_joint_obs"]    = b3_out["fica"].values

print("Subsidies computed.")
print(f"\nQuick check vs paper:")
print(f"  Fed sub pred  married: {df_out.loc[m,'fed_sub_pred'].mean():>10,.2f}  (paper: 122.41)")
print(f"  Fed sub pred  cohab:   {df_out.loc[c,'fed_sub_pred'].mean():>10,.2f}  (paper: 266.89)")
print(f"  Fed sub obs   married: {df_out.loc[m,'fed_sub_obs'].mean():>10,.2f}  (paper: 395.05)")
print(f"  Fed sub obs   cohab:   {df_out.loc[c,'fed_sub_obs'].mean():>10,.2f}  (paper: 231.80)")
print(f"  State sub pred married:{df_out.loc[m,'state_sub_pred'].mean():>10,.2f}  (paper: -54.21)")
print(f"  State sub pred cohab:  {df_out.loc[c,'state_sub_pred'].mean():>10,.2f}  (paper: -10.06)")

Subsidies computed.

Quick check vs paper:
  Fed sub pred  married:     253.92  (paper: 122.41)
  Fed sub pred  cohab:       203.77  (paper: 266.89)
  Fed sub obs   married:     879.69  (paper: 395.05)
  Fed sub obs   cohab:       543.36  (paper: 231.80)
  State sub pred married:    -29.93  (paper: -54.21)
  State sub pred cohab:      -38.04  (paper: -10.06)


In [28]:
# ── Replicate Table 2 ─────────────────────────────────────────────

def ms(series, mask, fmt=".2f"):
    v = series[mask]
    return f"{v.mean():{fmt}}  ({v.std():{fmt}})"

rep_total    = df_out["r_incearn"] + df_out["sp_incearn"]
pos_rep_m    = m & (rep_total > 0)
pos_rep_c    = c & (rep_total > 0)
rep_split_m  = (df_out.loc[pos_rep_m, ["r_incearn","sp_incearn"]].max(axis=1)
                / rep_total[pos_rep_m])
rep_split_c  = (df_out.loc[pos_rep_c, ["r_incearn","sp_incearn"]].max(axis=1)
                / rep_total[pos_rep_c])
pos_pred_m   = m & (df_out["hat_c_incearn"] > 0)
pos_pred_c   = c & (df_out["hat_c_incearn"] > 0)
pred_split_m = df_out.loc[pos_pred_m, "hat_c_split"]
pred_split_c = df_out.loc[pos_pred_c, "hat_c_split"]

rows = [
    ("Positive earnings (reported)",
     ms((rep_total>0).astype(float), m, ".3f"),   "0.935 (0.246)",
     ms((rep_total>0).astype(float), c, ".3f"),   "0.935 (0.247)"),
    ("Positive earnings (predicted)",
     ms((df_out["hat_incearn_r"]+df_out["hat_incearn_sp"]>0).astype(float), m, ".3f"),
     "0.963 (0.188)",
     ms((df_out["hat_incearn_r"]+df_out["hat_incearn_sp"]>0).astype(float), c, ".3f"),
     "0.969 (0.172)"),
    ("Reported earnings",
     ms(rep_total, m, ",.2f"),   "125,286.76 (119,779.91)",
     ms(rep_total, c, ",.2f"),   "105,188.00 (105,191.59)"),
    ("Predicted earnings",
     ms(df_out["hat_c_incearn"], m, ",.2f"),   "110,729.40 (57,936.40)",
     ms(df_out["hat_c_incearn"], c, ",.2f"),   "102,952.54 (54,275.74)"),
    ("Reported earnings split",
     f"{rep_split_m.mean():.3f}  ({rep_split_m.std():.3f})",   "0.745 (0.200)",
     f"{rep_split_c.mean():.3f}  ({rep_split_c.std():.3f})",   "0.723 (0.174)"),
    ("Predicted earnings split",
     f"{pred_split_m.mean():.3f}  ({pred_split_m.std():.3f})",   "0.648 (0.197)",
     f"{pred_split_c.mean():.3f}  ({pred_split_c.std():.3f})",   "0.641 (0.181)"),
    ("Fed+state subsidy (reported)",
     ms(df_out["total_sub_obs"], m, ",.2f"),   "442.45 (5,116.62)",
     ms(df_out["total_sub_obs"], c, ",.2f"),   "263.79 (3,247.05)"),
    ("Fed+state subsidy (predicted)",
     ms(df_out["total_sub_pred"], m, ",.2f"),   "68.19 (2,218.99)",
     ms(df_out["total_sub_pred"], c, ",.2f"),   "256.82 (1,623.22)"),
    ("Fed subsidy (reported)",
     ms(df_out["fed_sub_obs"], m, ",.2f"),   "395.05 (4,563.36)",
     ms(df_out["fed_sub_obs"], c, ",.2f"),   "231.80 (3,055.28)"),
    ("Fed subsidy (predicted)",
     ms(df_out["fed_sub_pred"], m, ",.2f"),   "122.41 (1,896.07)",
     ms(df_out["fed_sub_pred"], c, ",.2f"),   "266.89 (1,427.33)"),
    ("State subsidy (reported)",
     ms(df_out["state_sub_obs"], m, ",.2f"),   "47.41 (974.14)",
     ms(df_out["state_sub_obs"], c, ",.2f"),   "31.99 (584.34)"),
    ("State subsidy (predicted)",
     ms(df_out["state_sub_pred"], m, ",.2f"),   "-54.21 (487.06)",
     ms(df_out["state_sub_pred"], c, ",.2f"),   "-10.06 (332.98)"),
]

w = 42
print(f"{'TABLE 2':^126}")
print(f"{'':>{w}} {'Replication (Married)':>28} {'Paper (Married)':>24} "
      f"{'Replication (Cohab)':>28} {'Paper (Cohab)':>22}")
print("─" * 126)
for label, ym, pm, yc, pc in rows:
    print(f"{label:<{w}} {ym:>28} {pm:>24} {yc:>28} {pc:>22}")
print("─" * 126)
print(f"{'Observations':<{w}} {m.sum():>28,} {'16,098':>24} "
      f"{c.sum():>28,} {'21,136':>22}")

                                                           TABLE 2                                                            
                                                  Replication (Married)          Paper (Married)          Replication (Cohab)          Paper (Cohab)
──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
Positive earnings (reported)                             0.971  (0.168)            0.935 (0.246)               0.971  (0.167)          0.935 (0.247)
Positive earnings (predicted)                            0.990  (0.099)            0.963 (0.188)               0.994  (0.077)          0.969 (0.172)
Reported earnings                              125,125.11  (119,751.06)  125,286.76 (119,779.91)     104,319.22  (104,848.01) 105,188.00 (105,191.59)
Predicted earnings                              115,730.02  (49,401.20)   110,729.40 (57,936.40)      108,428.94  (46,770.59) 102,952.54 (54,275.74)


In [29]:
# ── Subsidy by year ───────────────────────────────────────────────
print("=== Mean predicted total subsidy by year ===")
piv = df_out.groupby(["year","sscouple_mar"])["total_sub_pred"].mean().unstack()
piv.columns = ["Cohabiting", "Married"]
piv["Married - Cohabiting"] = piv["Married"] - piv["Cohabiting"]
print(piv.round(1).to_string())

print("\n=== Mean observed total subsidy by year ===")
piv2 = df_out.groupby(["year","sscouple_mar"])["total_sub_obs"].mean().unstack()
piv2.columns = ["Cohabiting", "Married"]
piv2["Married - Cohabiting"] = piv2["Married"] - piv2["Cohabiting"]
print(piv2.round(1).to_string())

=== Mean predicted total subsidy by year ===
      Cohabiting  Married  Married - Cohabiting
year                                           
2012       134.9    231.1                  96.2
2013       134.3    275.7                 141.4
2014       189.8    233.0                  43.3
2015       173.8    225.0                  51.3
2016       186.5    224.6                  38.0
2017       182.9    192.3                   9.4

=== Mean observed total subsidy by year ===
      Cohabiting  Married  Married - Cohabiting
year                                           
2012       761.6   1129.2                 367.6
2013       728.9   1081.1                 352.1
2014       632.7   1034.0                 401.3
2015       620.3   1228.8                 608.5
2016       524.5    999.6                 475.1
2017       448.0    914.7                 466.7


In [30]:
# ── Save ──────────────────────────────────────────────────────────
new_cols = [
    "fed_sub_pred",  "state_sub_pred", "total_sub_pred",
    "fed_sub_obs",   "state_sub_obs",  "total_sub_obs",
    "fiitax_joint_pred", "siitax_joint_pred", "fica_joint_pred",
    "fiitax_joint_obs",  "siitax_joint_obs",  "fica_joint_obs",
]
df_out.to_pickle(taxsim_output)
print(f"Saved → {taxsim_output}")
print(f"Shape: {df_out.shape}")
print(f"New columns: {new_cols}")

Saved → /Users/guypigott/python-venv-demo/EDS/Data/Output/acs_ssc_taxsim_results_second.pkl
Shape: (37907, 84)
New columns: ['fed_sub_pred', 'state_sub_pred', 'total_sub_pred', 'fed_sub_obs', 'state_sub_obs', 'total_sub_obs', 'fiitax_joint_pred', 'siitax_joint_pred', 'fica_joint_pred', 'fiitax_joint_obs', 'siitax_joint_obs', 'fica_joint_obs']


In [31]:
import scipy.linalg as la
import warnings
warnings.filterwarnings('ignore')

from linearmodels.iv import IV2SLS
from statsmodels.tools import add_constant as sm_add_const

#IN_PKL = "/Users/trinityrose/Desktop/acs_ssc_msub.pkl"

df_reg = pd.read_pickle(taxsim_output)
print(f"Loaded: {df.shape}")
print(f"Married share: {df['sscouple_mar'].mean():.3f}")

Loaded: (37907, 72)
Married share: 0.426


In [32]:
# Cell 2 — Prepare variables
df_reg["married"]        = df_reg["sscouple_mar"].astype(float)
df_reg["msub_obs_k"]     = df_reg["total_sub_obs"]  / 1000    # was msub_total_obs
df_reg["msub_hat_k"]     = df_reg["total_sub_pred"] / 1000    # was msub_total_hat
df_reg["legal_marriage"] = df_reg["staterecog_policy"].astype(float)
df_reg["male"]         = df_reg["r_male"].astype(float)
df_reg["any_children"] = df_reg["c_anychildren"].astype(float)
df_reg["n_children"]   = df_reg["c_children"].astype(float)
df_reg["age_older"]    = df_reg["c_agemax"].astype(float)
df_reg["age_diff"]     = df_reg["c_agediff"].astype(float)
df_reg["edu_max"]      = df_reg["c_edumax"].astype(float)
df_reg["edu_diff"]     = df_reg["c_edudiff"].astype(float)
df_reg["same_race"]    = df_reg["c_racecomp"].astype(float)
df_reg["medicaid_exp"] = df_reg["medicaid_exp"].astype(float)
df_reg["earn_split_obs"] = df_reg["c_incearn_split"].astype(float)
df_reg["earn_split_hat"] = df_reg["hat_c_split"].astype(float)

for i in range(1, 6):
    df_reg[f"hat_earn{i}"]  = df_reg[f"hat_c_incearn{i}"].astype(float)
    df_reg[f"hat_split{i}"] = df_reg[f"hat_c_split{i}"].astype(float)

df_reg["year_fe"]  = df_reg["year"].astype(str)
df_reg["state_fe"] = df_reg["statefip"].astype(str)

key_cols = ["married", "msub_obs_k", "msub_hat_k", "legal_marriage",
            "male", "any_children", "n_children", "age_older", "age_diff",
            "edu_max", "edu_diff", "same_race", "medicaid_exp",
            "earn_split_obs", "earn_split_hat"]
df_reg = df_reg.dropna(subset=key_cols).copy()
print(f"Analysis sample: {len(df_reg):,}")
print(f"Married share:   {df_reg['married'].mean():.3f}  (paper: 0.432)")

Analysis sample: 37,907
Married share:   0.426  (paper: 0.432)


In [33]:
# Cell 3 — Helpers
def make_dummies(df, col):
    return pd.get_dummies(df[col], prefix=col, drop_first=True, dtype=float)

#def drop_collinear(X, tol=1e-8):
   # """Remove columns causing rank deficiency via QR with column pivoting."""
   # arr = X.values.astype(float)
   # Q, R, P = la.qr(arr, pivoting=True)
   # diag   = np.abs(np.diag(R))
   # rank   = int(np.sum(diag > tol * diag[0]))
   # keep   = sorted(P[:rank])
   # return X.iloc[:, keep]

def clean_design_matrix(X: pd.DataFrame) -> pd.DataFrame:
    # drop constant columns
    X = X.loc[:, X.nunique(dropna=False) > 1]

    # drop exact duplicate columns
    X = X.T.drop_duplicates().T

    return X

def run_spec(df, extra_controls=None, iv=False):
    extra_controls = extra_controls or []
    base = ["legal_marriage", "medicaid_exp", "male",
            "any_children", "n_children",
            "age_older", "age_diff",
            "edu_max", "edu_diff", "same_race"]

    year_dums = make_dummies(df, "year_fe")
    state_dums = make_dummies(df, "state_fe")

    X_cols = base + extra_controls
    X = pd.concat([df[X_cols], year_dums, state_dums], axis=1).astype(float)
    X = sm_add_const(X)

    # add observed subsidy for OLS spec
    if not iv:
        X.insert(1, "msub_obs_k", df["msub_obs_k"].values)

    # important: remove constant / duplicate columns
    X = clean_design_matrix(X)

    y = df["married"]

    if not iv:
        res = IV2SLS(y, X, None, None).fit(cov_type="robust")
    else:
        endog = df[["msub_obs_k"]]
        instr = df[["msub_hat_k"]]
        res = IV2SLS(y, X, endog, instr).fit(cov_type="robust")

    return res

def first_stage_info(res):
    try:
        fs    = res.first_stage
        indiv = fs.individual["msub_obs_k"]
        c     = float(indiv.params["msub_hat_k"])
        se    = float(indiv.std_errors["msub_hat_k"])
        p     = float(indiv.pvalues["msub_hat_k"])
        f     = float(fs.diagnostics.loc["msub_obs_k", "f.stat"])
        return c, se, p, f
    except Exception as e:
        print(f"first_stage_info error: {e}")
        return np.nan, np.nan, np.nan, np.nan

print("Helpers defined.")

Helpers defined.


In [34]:
# Cell 4 — Run all 6 columns
poly_cols = ([f"hat_earn{i}" for i in range(2, 6)] +
             [f"hat_split{i}" for i in range(2, 6)])

print("Col 1: OLS no income controls...")
col1 = run_spec(df_reg, iv=False)

print("Col 2: IV  no income controls...")
col2 = run_spec(df_reg, iv=True)

print("Col 3: OLS + earnings split...")
col3 = run_spec(df_reg, extra_controls=["earn_split_obs"], iv=False)

print("Col 4: IV  + earnings split...")
col4 = run_spec(df_reg, extra_controls=["earn_split_hat"], iv=True)

print("Col 5: OLS + poly + control function...")
col5 = run_spec(df_reg, extra_controls=poly_cols, iv=False)

print("Col 6: IV  + poly + control function...")
col6 = run_spec(df_reg, extra_controls=poly_cols, iv=True)

print("All columns estimated.")

Col 1: OLS no income controls...


Col 2: IV  no income controls...
Col 3: OLS + earnings split...
Col 4: IV  + earnings split...
Col 5: OLS + poly + control function...
Col 6: IV  + poly + control function...
All columns estimated.


In [35]:
# Cell 5 — Display table
def stars(p):
    if p < 0.01: return "***"
    if p < 0.05: return "**"
    if p < 0.10: return "*"
    return "   "

def get_coef_se_p(res, name):
    if name not in res.params.index:
        return None, None, None
    return res.params[name], res.std_errors[name], res.pvalues[name]

def first_stage_info(res):
    try:
        indiv = res.first_stage.individual["msub_obs_k"]
        c  = float(indiv.params["msub_hat_k"])
        se = float(indiv.std_errors["msub_hat_k"])
        p  = float(indiv.pvalues["msub_hat_k"])
        f  = float(res.first_stage.diagnostics.loc["msub_obs_k", "f.stat"])
        return c, se, p, f
    except Exception as e:
        print(f"first_stage_info error: {e}")
        return np.nan, np.nan, np.nan, np.nan

results  = [col1, col2, col3, col4, col5, col6]
is_iv    = [False, True, False, True, False, True]
mean_dv  = df_reg["married"].mean()

sep = "-" * 105

print("=" * 105)
print("TABLE 3 — Effect of Marriage Subsidy ($1,000s) on P(Married)")
print("=" * 105)
hdr = f"{'':42}" + "".join(f"{'OLS' if not v else 'IV':>11}" for v in is_iv)
print(hdr)
print(sep)

def print_row(label, varname, paper_vals):
    # Coefficient row
    line = f"{label:42}"
    for res in results:
        c, se, p = get_coef_se_p(res, varname)
        if c is not None:
            line += f"  {c:7.3f}{stars(p)}"
        else:
            line += f"  {'—':>10}"
    print(line)
    # SE row
    line = f"{'':42}"
    for res in results:
        c, se, p = get_coef_se_p(res, varname)
        if se is not None:
            line += f"  ({se:6.3f})  "
        else:
            line += f"  {'':>10}"
    print(line)
    # Paper row
    print(f"  {'(paper)':40}" + "".join(f"  {v:>10}" for v in paper_vals))
    print()

print_row("Marriage subsidy ($1,000s)", "msub_obs_k",
          ["0.005***", "0.008***", "0.004***", "0.009*  ", "0.005***", "0.014***"])
print_row("Legal marriage", "legal_marriage",
          ["0.066***", "0.066***", "0.066***", "0.066***", "0.116***", "0.116***"])
print_row("State Medicaid expansion", "medicaid_exp",
          ["0.010   ", "0.010   ", "0.009   ", "0.010   ", "0.035***", "0.034***"])

print(sep)

# First stage
print(f"{'First stage coef':42}", end="")
for res, iv in zip(results, is_iv):
    if iv:
        c, se, p, f = first_stage_info(res)
        print(f"  {c:7.3f}{stars(p)}", end="")
    else:
        print(f"  {'—':>10}", end="")
print()

print(f"{'':42}", end="")
for res, iv in zip(results, is_iv):
    if iv:
        c, se, p, f = first_stage_info(res)
        print(f"  ({se:6.3f})  ", end="")
    else:
        print(f"  {'':>10}", end="")
print()
print(f"  {'(paper)':40}"
      + f"  {'—':>10}  {'0.463***':>10}  {'—':>10}  {'0.408***':>10}  {'—':>10}  {'0.420***':>10}")
print()

print(f"{'First stage F-stat':42}", end="")
for res, iv in zip(results, is_iv):
    if iv:
        _, _, _, f = first_stage_info(res)
        print(f"  {f:>10.1f}", end="")
    else:
        print(f"  {'—':>10}", end="")
print()
print(f"  {'(paper)':40}"
      + f"  {'—':>10}  {'[474.7]':>10}  {'—':>10}  {'[221.0]':>10}  {'—':>10}  {'[261.3]':>10}")

print(sep)
print(f"{'Mean dep var':42}" + "".join(f"  {mean_dv:>10.3f}" for _ in results))
print(f"{'N':42}" + "".join(f"  {int(res.nobs):>10,}" for res in results))
print(f"  {'(paper N)':40}" + "".join(f"  {'37,234':>10}" for _ in results))
print("=" * 105)

TABLE 3 — Effect of Marriage Subsidy ($1,000s) on P(Married)
                                                  OLS         IV        OLS         IV        OLS         IV
---------------------------------------------------------------------------------------------------------
Marriage subsidy ($1,000s)                    0.008***    0.009       0.008***    0.051***    0.008***    0.005   
                                            ( 0.001)    ( 0.025)    ( 0.001)    ( 0.018)    ( 0.001)    ( 0.035)  
  (paper)                                     0.005***    0.008***    0.004***    0.009*      0.005***    0.014***

Legal marriage                                0.069***    0.069***    0.069***    0.074***    0.065***    0.065***
                                            ( 0.009)    ( 0.010)    ( 0.009)    ( 0.010)    ( 0.009)    ( 0.010)  
  (paper)                                     0.066***    0.066***    0.066***    0.066***    0.116***    0.116***

State Medicaid expansion        

In [36]:
# Cell 6 — Elasticity
c6, _, _ = get_coef_se_p(col6, "msub_obs_k")
mean_msub = df_reg["msub_obs_k"].mean()
elasticity = c6 * (mean_msub / mean_dv)
print(f"IV coef (col 6):     {c6:.4f}")
print(f"Mean subsidy ($k):   {mean_msub:.3f}")
print(f"Mean married:        {mean_dv:.3f}")
print(f"Implied elasticity:  {elasticity:.4f}  (paper: 0.011)")

IV coef (col 6):     0.0052
Mean subsidy ($k):   0.806
Mean married:        0.426
Implied elasticity:  0.0098  (paper: 0.011)
